# From DN to reflectance: Landsat 8 over Prague

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/dn_to_reflectance_en.ipynb)

**OpenSkyLab · notebook for the visual** [From DN to reflectance](https://openskylab.space/edu_viz_dn_to_reflectance.html) · [česká verze](dn_to_reflectance_cs.ipynb)

The visual walks through the chain from digital number to radiance to top-of-atmosphere (TOA) and surface (BOA) reflectance and shows what dark object subtraction (DOS) and the empirical line correct. Here you compute the whole chain on two real Landsat 8 images of Prague and compare the result with Sentinel-2 L2A reflectance from the same week.

**Goal:** convert DN to radiance and TOA reflectance, show why DN from two days are not comparable, measure the atmospheric effect over water, and compare DOS and the empirical line with surface reflectance.

**Data:** Landsat 8 OLI Collection 1 Level-1 from the Google Cloud public archive: `LC08_L1TP_191025_20190626_20190705_01_T1` (26 June 2019, sun elevation 60°) and `LC08_L1TP_191025_20190218_20190222_01_T1` (18 Feb 2019, sun elevation 26°), bands B1–B7, BQA and MTL. For comparison Sentinel-2 L2A `S2B_33UVR_20190625_0_L2A` (25 June 2019) from Earth Search. A 12 × 12 km window around central Prague, about 25 MB.

**Tasks:** five. Write your answers in the *Your answer* cells below the tasks. Lines marked `# ◀ parameter` are meant to be changed; run the cell again with Shift + Enter.

**Time:** about 45 min. You need a Google account (Colab); nothing is installed on your computer. Run the cells in order from top to bottom.

## 1. Setup and data

In [ ]:
%pip install -q rasterio

In [ ]:
import requests
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling

GCS = "https://storage.googleapis.com/gcp-public-data-landsat/LC08/01/191/025/"
SCENES = {"summer": "LC08_L1TP_191025_20190626_20190705_01_T1", "winter": "LC08_L1TP_191025_20190218_20190222_01_T1"}
X0, Y0, SIZE = 452490, 5541690, 12000          # lower-left corner of the window in UTM 33N (m) and its side
BANDS = {1: "coastal 443", 2: "blue 482", 3: "green 561", 4: "red 655", 5: "NIR 865", 6: "SWIR1 1609", 7: "SWIR2 2201"}

def mtl(scene):
    txt = requests.get(f"{GCS}{scene}/{scene}_MTL.txt", timeout=60).text
    return {k.strip(): v.strip().strip('"') for k, v in (l.split(" = ") for l in txt.splitlines() if " = " in l)}

def read(scene, band):
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY=5, GDAL_HTTP_RETRY_DELAY=2):
        with rasterio.open(f"/vsicurl/{GCS}{scene}/{scene}_{band}.TIF") as r:
            w = rasterio.windows.from_bounds(X0, Y0, X0 + SIZE, Y0 + SIZE, r.transform).round_offsets().round_lengths()
            read.transform = r.window_transform(w)
            return r.read(1, window=w).astype(float)

L8 = {}
for season, scene in SCENES.items():
    meta = mtl(scene)
    dn = {b: read(scene, f"B{b}") for b in BANDS}
    L8_transform = read.transform
    qa = read(scene, "BQA").astype(int)
    ok = ((qa >> 4) & 1 == 0) & ((qa >> 7) & 3 < 3) & (dn[1] > 0)     # no clouds and no high-confidence shadows
    L8[season] = {"meta": meta, "dn": dn, "ok": ok, "transform": L8_transform}
    print(f"{season}: {scene}, sun elevation {float(meta['SUN_ELEVATION']):.1f}°, Earth–Sun distance {meta['EARTH_SUN_DISTANCE']} AU, clear pixels {ok.mean():.1%}")

## 2. From DN to radiance and TOA reflectance

Radiance: L = M_L · DN + A_L. TOA reflectance: ρ = (M_ρ · DN + A_ρ) / sin(sun elevation). The coefficients are in the MTL file; M_ρ already includes the Earth–Sun distance and the solar irradiance of the band.

In [ ]:
for s in L8.values():
    m, dn = s["meta"], s["dn"]
    sun = np.sin(np.radians(float(m["SUN_ELEVATION"])))
    s["rad"] = {b: float(m[f"RADIANCE_MULT_BAND_{b}"]) * dn[b] + float(m[f"RADIANCE_ADD_BAND_{b}"]) for b in BANDS}
    s["toa"] = {b: (float(m[f"REFLECTANCE_MULT_BAND_{b}"]) * dn[b] + float(m[f"REFLECTANCE_ADD_BAND_{b}"])) / sun for b in BANDS}

rows = []
for b, name in BANDS.items():
    r = {"band": f"B{b} {name}"}
    for season, s in L8.items():
        ok = s["ok"]
        r[f"DN {season}"] = np.median(s["dn"][b][ok])
        r[f"radiance {season}"] = np.median(s["rad"][b][ok])
        r[f"ρ TOA {season}"] = np.median(s["toa"][b][ok])
    rows.append(r)
tab = pd.DataFrame(rows).set_index("band")
tab["ratio DN summer/winter"] = tab["DN summer"] / tab["DN winter"]
tab["ratio ρ summer/winter"] = tab["ρ TOA summer"] / tab["ρ TOA winter"]
tab.round(3)

> **✏️ Task 1.** How much do the median DN differ between summer and winter, and how much the median TOA reflectance? How much of the DN difference is explained by sun elevation alone (compare the ratio of the sines)? Why does even TOA reflectance not agree completely between the dates (vegetation, snow, atmosphere)?

*Your answer:* 

## 3. Surface reflectance from Sentinel-2 as a reference

Sentinel-2 L2A from 25 June 2019 (one day earlier) has BOA reflectance from the Sen2Cor model. We resample it to the Landsat grid (average into 30 m pixels) and pair bands of similar wavelength. Do not expect a perfect match: it is a different sensor with slightly different bands, an image taken one day earlier and from a different viewing angle.

In [ ]:
item = requests.get("https://earth-search.aws.element84.com/v1/collections/sentinel-2-l2a/items/S2B_33UVR_20190625_0_L2A", timeout=60).json()
PAIR = {1: "coastal", 2: "blue", 3: "green", 4: "red", 5: "nir08", 6: "swir16", 7: "swir22"}
shape = L8["summer"]["dn"][1].shape
grid = L8["summer"]["transform"]
def s2(key, res=Resampling.average):
    out = np.zeros(shape, np.float32)
    with rasterio.Env(GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR", GDAL_HTTP_MAX_RETRY=5, GDAL_HTTP_RETRY_DELAY=2):
        with rasterio.open("/vsicurl/" + item["assets"][key]["href"]) as r:
            reproject(rasterio.band(r, 1), out, dst_transform=grid, dst_crs="EPSG:32633", resampling=res)
    return out
boa = {b: s2(k) / 10000 for b, k in PAIR.items()}          # processing baseline 02.12: no DN offset
scl = s2("scl", Resampling.mode)
summer = L8["summer"]
valid = summer["ok"] & np.isin(scl, [4, 5, 6]) & (boa[1] > 0)
print(f"pixels for comparison: {valid.sum()}")
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
rgb = lambda d: np.clip(np.dstack([d[4], d[3], d[2]]) / 0.25, 0, 1)
ax[0].imshow(rgb(summer["toa"])); ax[0].set_title("Landsat 8, TOA reflectance, 26 June 2019")
ax[1].imshow(rgb(boa)); ax[1].set_title("Sentinel-2 L2A, BOA reflectance, 25 June 2019")
for a in ax: a.axis("off")
plt.show()

> **✏️ Task 2.** Compare the two images with the same colour stretch. Why does the TOA reflectance image look brighter, bluer and less contrasting? Which surfaces change most between TOA and BOA, dark or bright ones? Explain it with the additive contribution of the atmosphere.

*Your answer:* 

## 4. Blue water above the atmosphere

The Vltava and reservoirs are almost black in the NIR. We compare their TOA and BOA reflectance in all bands: the difference is mainly light scattered by the atmosphere (path radiance).

In [ ]:
ndwi = (boa[3] - boa[5]) / (boa[3] + boa[5])
water = valid & (ndwi > 0.3) & (scl == 6)
w = pd.DataFrame({"ρ TOA (Landsat)": [np.median(summer["toa"][b][water]) for b in BANDS],
                  "ρ BOA (Sentinel-2)": [np.median(boa[b][water]) for b in BANDS]}, index=[f"B{b} {n}" for b, n in BANDS.items()])
w["TOA − BOA"] = w.iloc[:, 0] - w.iloc[:, 1]
print(f"water pixels: {water.sum()}")
w.round(4)

> **✏️ Task 3.** How does TOA − BOA over water change with wavelength? Compare the ratio between bands B1 and B4 with Rayleigh's law (scattering ∝ λ⁻⁴). Why is water blue above the atmosphere although it is dark in all bands at the surface?

*Your answer:* 

## 5. Dark object subtraction (DOS)

We assume that the darkest pixels in the scene have almost zero surface reflectance (Chavez 1988 assumes 1 %). Whatever they have on top is path radiance, which we subtract from the whole scene. DOS corrects only the additive part of the atmosphere.

In [ ]:
pct = 0.1          # ◀ parameter: percentile taken as the dark object (%)
rho_dark = 0.01    # ◀ parameter: assumed true reflectance of the dark object
dos = {b: summer["toa"][b] - (np.percentile(summer["toa"][b][summer["ok"]], pct) - rho_dark) for b in BANDS}

def rmse(a, b, m=valid): return float(np.sqrt(np.mean((a[m] - b[m]) ** 2)))
def bias(a, b, m=valid): return float(np.mean(a[m] - b[m]))
res = pd.DataFrame({
    "dark object ρ TOA": [np.percentile(summer["toa"][b][summer["ok"]], pct) for b in BANDS],
    "RMSE TOA": [rmse(summer["toa"][b], boa[b]) for b in BANDS],
    "RMSE DOS": [rmse(dos[b], boa[b]) for b in BANDS],
    "bias DOS": [bias(dos[b], boa[b]) for b in BANDS],
}, index=[f"B{b}" for b in BANDS])
res.round(4)

> **✏️ Task 4.** In which bands does DOS reduce the error against BOA most and in which does it change almost nothing? Set `pct = 2` and then `rho_dark = 0`: what happens? When does DOS fail completely (a winter scene with snow and no dark objects, haze over only part of the scene)?

*Your answer:* 

## 6. Empirical line

In the field a dark and a bright panel with measured reflectance are placed in the image. Here the panels are dark and bright pixels whose surface reflectance we know from Sentinel-2. For each band we fit a line ρ_BOA = a · ρ_TOA + b: the offset b corresponds to path radiance, the slope a to atmospheric transmittance.

In [ ]:
n_panel = 300        # ◀ parameter: how many darkest and brightest pixels to use as panels
rng = np.random.default_rng(0)
el, rows = {}, []
for b in BANDS:
    v = np.flatnonzero(valid.ravel())
    order = v[np.argsort(boa[b].ravel()[v])]
    panels = np.r_[order[:n_panel], order[-n_panel:]]               # dark and bright "panels"
    a, c = np.polyfit(summer["toa"][b].ravel()[panels], boa[b].ravel()[panels], 1)
    el[b] = a * summer["toa"][b] + c
    rows.append({"band": f"B{b}", "slope a": a, "offset b": c, "RMSE TOA": rmse(summer["toa"][b], boa[b]),
                 "RMSE DOS": rmse(dos[b], boa[b]), "RMSE emp. line": rmse(el[b], boa[b])})
print(pd.DataFrame(rows).set_index("band").round(4))
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
sel = rng.choice(np.flatnonzero(valid.ravel()), 4000, replace=False)
for a_, (name, est) in zip(ax, [("TOA", summer["toa"]), ("DOS", dos), ("empirical line", el)]):
    for b, col in [(2, "tab:blue"), (4, "tab:red"), (5, "tab:purple")]:
        a_.scatter(boa[b].ravel()[sel], est[b].ravel()[sel], s=2, alpha=0.4, c=col, label=f"B{b}")
    a_.axline((0, 0), slope=1, color="k", lw=0.8); a_.set(xlim=(0, 0.5), ylim=(0, 0.5), xlabel="ρ BOA Sentinel-2", ylabel=name, title=name)
ax[0].legend(); plt.tight_layout(); plt.show()

> **✏️ Task 5.** Why does the empirical line have a negative offset and a slope different from 1 in the blue bands? What do they express physically? Which method did best and why can DOS not correct the slope? Set `n_panel = 20`: what happens to the reliability of the line? Why are two panels enough in the field, but must be in the same image and under the same illumination?

*Your answer:* 

---
### Data sources and references

- Landsat 8 OLI Collection 1 Level-1, U.S. Geological Survey; Google Cloud public archive (`gcp-public-data-landsat`). USGS data have no use restrictions, attribution requested.
- USGS (2019). *Landsat 8 (L8) Data Users Handbook*, version 5.0, ch. 5 (DN to radiance and TOA reflectance).
- Copernicus Sentinel-2 data 2019, ESA (L2A, Sen2Cor), AWS Open Data registry (`sentinel-cogs`), Earth Search catalogue (Element 84). Free use with attribution.
- Chavez, P. S. (1988). An improved dark-object subtraction technique for atmospheric scattering correction of multispectral data. *Remote Sensing of Environment* 24, 459–479. https://doi.org/10.1016/0034-4257(88)90019-3
- Smith, G. M., Milton, E. J. (1999). The use of the empirical line method to calibrate remotely sensed data to reflectance. *International Journal of Remote Sensing* 20, 2653–2662. https://doi.org/10.1080/014311699211994

### How to cite

OpenSkyLab (2026). From DN to reflectance: Landsat 8 over Prague [notebook, teaching resource], version 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/dn_to_reflectance_en.ipynb. Licence CC BY 4.0.

Notebook text and code: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data are under their providers' licences (see above).

*This resource was developed by the OpenSkyLab team with the assistance of the generative AI Claude (Anthropic), used for drafting code and text. The authors are responsible for the content.*